# Module 2 · Lesson 04: The PCTF Framework

**PCTF = Persona · Context · Task · Format**

A systematic approach to writing effective prompts. Instead of guessing,
use this framework to structure every prompt.

## What you will learn
1. The four PCTF components
2. Real-world examples (code review, documentation)
3. Reusable **prompt templates**
4. Adding **negative constraints**
5. Building an interactive PCTF builder

In [1]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / "module_02_prompt_enginnering/.env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
OPENAI_MODEL = "gpt-4o-mini" 
if client:
    print(f"OpenAI client ready - using model {OPENAI_MODEL}")

OpenAI client ready - using model gpt-4o-mini


In [2]:
def ask_open_ai(prompt: str, system_content: str= None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model:  str="gpt-4o-mini") -> str:
    """
    Send a prompt to the OpenAI Chat Completions API and return the assistant's reply.

    This helper function builds a chat message list, optionally including a system
    instruction, sends it to the specified model, and returns the generated text
    content from the first response choice.

    Args:
        prompt (str): The user prompt to send to the model.
        system_content (str, optional): Optional system-level instruction that
            defines the assistant's behavior, tone, or constraints. If provided,
            it is added as the first message in the conversation. Defaults to None.
        temperature (float, optional): Sampling temperature used to control
            randomness in the model's response. Lower values produce more
            deterministic outputs; higher values produce more creative or varied
            outputs. Defaults to 0.7.
        max_resp_tokens (int, optional): Maximum number of tokens allowed in the
            generated response. Defaults to 800.
        ai_model (str, optional): The model name to use for the request.
            Defaults to "gpt-4o-mini".

    Returns:
        str: The text content of the assistant's first response message.

    Raises:
        Exception: Propagates any exception raised by the OpenAI client request
            (for example, authentication errors, rate limits, invalid parameters,
            or network-related failures).

    Example of use:
         reply = ask("Explain recursion in simple terms.")
         print(reply)

        reply = ask(
             prompt="Write a haiku about Python.",
             system_content="You are a poetic assistant.",
             temperature=0.9,
             max_resp_tokens=100,
             ai_model="gpt-4o-mini"
        )        
        print(reply)
    """
    msgs = []
    if system_content:
        msgs.append({"role": "system", "content": system_content})
    msgs.append({"role":"user", "content": prompt})
    resp = client.chat.completions.create(
        model=ai_model,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_resp_tokens
    )
    return resp.choices[0].message.content

display(Markdown(f"**The function has run**"))

**The function has run**

---
## The PCTF Framework

| Component | Question | Example |
|-----------|----------|---------|
| **P**ersona | Who should the AI be? | "You are a senior code reviewer" |
| **C**ontext | What's the situation? | "We're building a FastAPI backend" |
| **T**ask | What should it do? | "Review this function for bugs" |
| **F**ormat | How should it respond? | "Use a numbered list with severity" |

---
## 1. Example: Code Review

In [3]:
system_prompt = '''You are senior Python developer with 10 years of experience.
You specialize in code quality, security and performance.
'''

user_prompt = '''Context: We're building a REST API with FastAPI. This function handles user login.

Task: Review this code for bugs, security issues and performance problems.

```python
def login(username, password):
    user = db.query(f"SELECT * FROM users WHERE username='{username}'")
    if user and user.password == password:
        token = str(random.randint(1000, 9999))
        return {"token": token}
    return {"error": "Invalid credentials"}
```
format: For each issue found, provide:
1. Issue severity (Critical, Warning, Suggestion).
2. Line reference.
3. Problem description.
4. Corrected code.
'''

result = ask_open_ai(user_prompt, system_content=system_prompt)
display(Markdown(result))

Here’s a review of the provided `login` function for issues related to bugs, security, and performance:

### Issue 1
1. **Severity:** Critical
2. **Line Reference:** `user = db.query(f"SELECT * FROM users WHERE username='{username}'")`
3. **Problem Description:** This line is vulnerable to SQL injection attacks because it directly interpolates user input (`username`) into the SQL query. An attacker could manipulate the `username` input to execute arbitrary SQL commands.
4. **Corrected Code:**
   ```python
   from sqlalchemy import select
   from sqlalchemy.orm import Session

   def login(username: str, password: str, db: Session):
       user = db.execute(select(User).where(User.username == username)).scalars().first()
   ```

### Issue 2
1. **Severity:** Critical
2. **Line Reference:** `if user and user.password == password:`
3. **Problem Description:** This line compares the plaintext password directly with the stored password, which is a security risk. Passwords should never be stored in plaintext; they should be hashed and compared using a secure method.
4. **Corrected Code:**
   ```python
   from passlib.context import CryptContext

   pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

   def login(username: str, password: str, db: Session):
       user = db.execute(select(User).where(User.username == username)).scalars().first()
       if user and pwd_context.verify(password, user.password):
   ```

### Issue 3
1. **Severity:** Suggestion
2. **Line Reference:** `token = str(random.randint(1000, 9999))`
3. **Problem Description:** Generating a token using a random integer is not secure, as it can be easily guessed. A better approach would be to use a library designed for secure token generation.
4. **Corrected Code:**
   ```python
   import secrets

   token = secrets.token_hex(16)  # Generates a secure random token
   ```

### Issue 4
1. **Severity:** Suggestion
2. **Line Reference:** `return {"error": "Invalid credentials"}`
3. **Problem Description:** The error message is too generic and could provide feedback to attackers about username enumeration. It is better to use a uniform message for invalid credentials.
4. **Corrected Code:**
   ```python
   return {"error": "Invalid username or password"}
   ```

### Final Revised Code
Here’s the complete revised `login` function incorporating all the corrections:

```python
from sqlalchemy import select
from sqlalchemy.orm import Session
from passlib.context import CryptContext
import secrets

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def login(username: str, password: str, db: Session):
    user = db.execute(select(User).where(User.username == username)).scalars().first()
    if user and pwd_context.verify(password, user.password):
        token = secrets.token_hex(16)  # Secure random token
        return {"token": token}
    return {"error": "Invalid username or password"}  # Uniform error message
```

### Summary
By addressing these issues, the `login` function not only becomes secure against common vulnerabilities but also adheres to better practices for handling user authentication.

---
## 2. Example: API Documentation

In [4]:
system_prompt = """You are a technical writer specializing in API documentation.
Write clear, concise docs that developers can use immediatly.
"""
user_prompt = """Context: We have a user management API built with FastAPI.

Task: Write API documentation for this endpoint.

```python
@app.post("/users")
def create_user(name: str, email: str, role: str = "viewer"):
    user = User(name=name, email=email, role=role)
    db.add(user)
    return {"id": user.id, "name": user.name}
```

Format: Include:
- Endpoint summary.
- Parameters table (name, type, required, description).
- Example request (curl).
- Example response (JSON).
- Possible error codes
"""

result = ask_open_ai(user_prompt, system_content=system_prompt, max_resp_tokens= 1200)
display(Markdown(result))

# User Management API Documentation

## Endpoint Summary
**POST** `/users`  
This endpoint allows the creation of a new user in the system. The user is assigned a role, with a default value of "viewer" if not specified.

---

## Parameters

| Parameter | Type   | Required | Description                          |
|-----------|--------|----------|--------------------------------------|
| `name`    | string | Yes      | The name of the user.                |
| `email`   | string | Yes      | The email address of the user.       |
| `role`    | string | No       | The role assigned to the user (default is "viewer"). |

---

## Example Request

### cURL
```bash
curl -X POST http://your-api-url/users \
-H "Content-Type: application/json" \
-d '{"name": "John Doe", "email": "john.doe@example.com", "role": "admin"}'
```

---

## Example Response

### JSON
```json
{
  "id": 1,
  "name": "John Doe"
}
```

---

## Possible Error Codes

- **400 Bad Request**: Returned if required parameters are missing or invalid.
- **409 Conflict**: Returned if the email already exists in the system.
- **500 Internal Server Error**: Returned for unexpected server errors.

Make sure to validate the input data before sending the request to ensure proper handling and response from the API.

---
## 3. Reusable Prompt Templates

In production, you build **template functions** for common tasks:

In [5]:
def create_pctf_prompt(persona: str, context: str, task: str, format_spec: str) -> str:
    """Build a structred PCTF prompt"""
    return f"""## Persona
{persona}

## Context
{context}

## Task
{task}

## Format
{format_spec}
"""

In [6]:
prompt = create_pctf_prompt(
    persona="You are a database expert with deep knowledge of SQL optimization.",
    context="We have a PostgreSQL database with 10M rows in the `orders` table. Queries are slow.",
    task="Suggest 3 specific optimizations for this query: SELECT * FROM orders WHERE status='pending' AND created_at > NOW() - INTERVAL '7 days' ORDER BY total DESC",
    format_spec="For each optimization: title, explanation, SQL example, expected improvement."
)
 
result = ask_open_ai(prompt, max_resp_tokens=600)
display(Markdown(result))

### Optimization 1: Create an Index on Status and Created At

**Explanation**: Creating a composite index on the `status` and `created_at` columns can significantly speed up the filtering process of the query. This will allow PostgreSQL to quickly locate the rows that meet the criteria without scanning the entire table.

**SQL Example**:
```sql
CREATE INDEX idx_orders_status_created_at ON orders (status, created_at);
```

**Expected Improvement**: With this index in place, the query planner can use it to quickly narrow down the results to only those rows where `status` is 'pending' and `created_at` falls within the last 7 days. This can reduce the query execution time from several seconds to milliseconds, especially as the table size grows.

---

### Optimization 2: Avoid SELECT *

**Explanation**: Instead of using `SELECT *`, specify only the columns that are needed for your application. This reduces the amount of data that needs to be retrieved and sent over the network, which can improve performance. 

**SQL Example**:
```sql
SELECT id, total, created_at FROM orders WHERE status='pending' AND created_at > NOW() - INTERVAL '7 days' ORDER BY total DESC;
```

**Expected Improvement**: By reducing the amount of data transferred, the query can execute faster and consume less memory. This can lead to performance improvements, especially if the `orders` table contains large text fields or binary data.

---

### Optimization 3: Use a Materialized View for Frequent Queries

**Explanation**: If this query is run frequently, consider creating a materialized view that pre-aggregates the data. This view can be refreshed periodically, which would allow for quick access to the most relevant results without processing the entire table each time.

**SQL Example**:
```sql
CREATE MATERIALIZED VIEW recent_pending_orders AS 
SELECT * FROM orders WHERE status='pending' AND created_at > NOW() - INTERVAL '7 days' ORDER BY total DESC;

-- To refresh the view periodically
REFRESH MATERIALIZED VIEW recent_pending_orders;
```

**Expected Improvement**: By querying the materialized view instead of the base table, you can significantly reduce query execution time. The materialized view will provide instant access to the filtered and sorted data, resulting in response times in the order of milliseconds instead of seconds. This is especially beneficial if the underlying data does not change frequently.

---
## 4. Negative Constraints

Telling the model what **NOT** to do is as important as what to do:

In [7]:
prompt = create_pctf_prompt(
    persona="You are senior developer mentoring a junior.",
    context="A junior developer submitted a pull request with this code.",
    task="""Review this code: 
```python
data = []
for i in range(len(items)):
    if items[i]['active'] == True:
        data.append(items[i]['name'].upper())
```""",
    format_spec="""Provide feedback. DO NOT:
- Be condescending or harsh.
- Rewrite the entire function.
- Use jargon without explanation.

DO:
- Explain WHY each change improves the code.
- Show improved version.
- Encourage what they did right."""

)

In [8]:
result = ask_open_ai(prompt, max_resp_tokens=1000)
display(Markdown(result))

Thank you for submitting your code! You've made a good start with filtering and transforming the `items` list based on their 'active' status. Here are some suggestions to improve your code:

### Feedback

1. **Use Direct Iteration**: Instead of using `range(len(items))`, you can iterate directly over `items`. This makes the code cleaner and more readable. It also avoids the need to access `items` by index.

2. **Boolean Comparison**: The condition `if items[i]['active'] == True` can be simplified to `if items[i]['active']`. In Python, the truthiness of a value is evaluated directly, so you don’t need to explicitly compare it to `True`.

3. **List Comprehension**: You can achieve the same result in a more concise way using a list comprehension. This not only shortens the code but also improves readability.

### Improved Version

Here’s how your code could look after these improvements:

```python
data = [item['name'].upper() for item in items if item['active']]
```

### Why These Changes Improve the Code

- **Direct Iteration**: It simplifies the loop structure and improves readability. When someone reads your code, they can immediately see that you're working with each item in `items`, rather than having to think about indexing.
  
- **Simplified Condition**: This makes your code easier to read and understand. It's immediately clear that you're checking for truthiness rather than doing an explicit comparison.

- **List Comprehension**: This is a Pythonic way to create lists. It condenses the logic into a single line which can often be easier to understand at a glance, while also being more efficient.

### What You Did Right

- You correctly identified the need to filter active items and transform their names to uppercase. Your approach is on the right track, and these improvements will help make your code cleaner and more efficient. Keep up the good work, and don’t hesitate to reach out if you have any questions about these suggestions!

---
## 5. Exercise — Build Your Own PCTF Prompt 🏋️

---
## Key Takeaways 📝

| Component | Tips |
|-----------|------|
| **Persona** | Be specific about expertise level and domain |
| **Context** | Include tech stack, constraints, audience |
| **Task** | One clear, specific action verb |
| **Format** | Tables, lists, JSON — be explicit |
| **Negative constraints** | Say what NOT to do to avoid common pitfalls |
| **Templates** | Build reusable functions for common tasks |

---
**Next:** `05_prompt_evaluation.ipynb` — Measure and compare prompt effectiveness